In [1]:
# Initial setup follows last notebook, max_leaf_nodes

In [2]:
import pandas as pd

In [3]:
from data.nbaDataImport import load_nba_games

In [27]:
nba_data = load_nba_games()

         id   game_date  season  home_team_id      home_team_name  \
0  21717856  2026-10-20    2026            20     New York Knicks   
1  21717857  2026-10-20    2026            27   San Antonio Spurs   
2  21717858  2026-10-21    2026            16          Miami Heat   
3  21717859  2026-10-21    2026            14  Los Angeles Lakers   
4  21717922  2026-10-21    2026            22       Orlando Magic   

   visitor_team_id       visitor_team_name  home_team_score  \
0               23      Philadelphia 76ers              100   
1               21   Oklahoma City Thunder              104   
2               18  Minnesota Timberwolves              106   
3               10   Golden State Warriors              127   
4                1           Atlanta Hawks              125   

   visitor_team_score  postseason  
0                 106       False  
1                 116       False  
2                 131       False  
3                 105       False  
4                 132     

In [5]:
nba_data.head()

,id,game_date,season,home_team_id,home_team_name,visitor_team_id,visitor_team_name,home_team_score,visitor_team_score,postseason
0,21717856,2026-10-20,2026,20,New York Knicks,23,Philadelphia 76ers,100,106,False
1,21717857,2026-10-20,2026,27,San Antonio Spurs,21,Oklahoma City Thunder,104,116,False
2,21717858,2026-10-21,2026,16,Miami Heat,18,Minnesota Timberwolves,106,131,False
3,21717859,2026-10-21,2026,14,Los Angeles Lakers,10,Golden State Warriors,127,105,False
4,21717922,2026-10-21,2026,22,Orlando Magic,1,Atlanta Hawks,125,132,False


In [28]:
nba_data.describe()

,id,season,home_team_id,visitor_team_id,home_team_score,visitor_team_score
count,5.452000e+03,5452.000000,5452.000000,5452.000000,5452.000000,5452.000000
mean,9.875866e+06,2023.578687,15.423147,15.415077,115.171313,113.055393
std,8.371928e+06,1.183359,8.577560,8.583727,12.644251,12.708429
min,8.573550e+05,2022.000000,1.000000,1.000000,73.000000,66.000000
25%,1.037636e+06,2023.000000,8.000000,8.000000,107.000000,104.000000
50%,1.590752e+07,2024.000000,15.000000,15.000000,115.000000,113.000000
75%,1.844695e+07,2025.000000,23.000000,23.000000,124.000000,122.000000
max,2.171809e+07,2026.000000,30.000000,30.000000,175.000000,176.000000


In [29]:


# rename game_date column to date for readability
nba_data = nba_data.rename(columns={"game_date":"date"})

# One row per game from the home team's perspective.
home = nba_data[
    ["id", "date", "season", "home_team_id",
     "home_team_score", "visitor_team_score"]
].rename(columns={
    "home_team_id": "team_id",
    "home_team_score": "points_for",
    "visitor_team_score": "points_against"
})

# The same games from the visiting team's perspective.
away = nba_data[
    ["id", "date", "season", "visitor_team_id",
     "visitor_team_score", "home_team_score"]
].rename(columns={
    "visitor_team_id": "team_id",
    "visitor_team_score": "points_for",
    "home_team_score": "points_against"
})

#setup home versus away bool
home["is_home"] = True
away["is_home"] = False


# Stack the tables vertically, similar to SQL UNION ALL.
team_games = pd.concat([home, away], ignore_index=True)

team_games["date"] = pd.to_datetime(team_games["date"], utc=True)

team_games = team_games.sort_values(
    ["team_id", "season", "date", "id"]
).reset_index(drop=True)

In [30]:
team_games["won"] = (
    team_games["points_for"] > team_games["points_against"]
).astype(int)

team_games["previous_game_won"] = (
    team_games
    .groupby(["team_id", "season"])["won"]
    .shift(1)
)

In [31]:
# transformation for improved model features

In [32]:
team_games["date"] = pd.to_datetime(team_games["date"], utc=True)
team_games = team_games.sort_values(
    ["team_id", "season", "date", "id"]
).reset_index(drop=True)

team_games["point_differential"] = (
    team_games["points_for"] - team_games["points_against"]
)

groups = team_games.groupby(["team_id", "season"])

#win percentages BEFORE this game.
groups = team_games.groupby(["team_id", "season"])

team_games["season_win_percentage"] = groups["won"].transform(
    lambda s: s.shift(1).expanding().mean()
)

team_games["win_percentage_last_five_games"] = groups["won"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean()
)

# Calendar days between game dates: back-to-back games = 1.
previous_date = groups["date"].shift(1)
team_games["rest_days_since_last_game"] = (
    team_games["date"].dt.normalize() -
    previous_date.dt.normalize()
).dt.days

# Average scoring margin over ALL earlier games this season.
team_games["average_point_differential"] = (
    groups["point_differential"].transform(
        lambda s: s.shift(1).expanding().mean()
    ))

for source, prefix in [
    ("points_for", "points_scored"),
    ("points_against", "points_allowed"),
]:
    team_games[f"{prefix}_per_game"] = groups[source].transform(
        lambda s: s.shift(1).expanding().mean()
    )
    team_games[f"{prefix}_last_five_games"] = groups[source].transform(
        lambda s: s.shift(1).rolling(5, min_periods=5).mean()
    )


In [33]:
# setup home and visitor records
# Requires completed games with valid final scores.
# team_games is already sorted chronologically within each team/season.
for venue, venue_mask in [
    ("home", team_games["is_home"]),
    ("visitor", ~team_games["is_home"]),
]:
    # Indicators are zero for games played at the other venue.
    venue_games = venue_mask.astype(int)
    venue_wins = venue_games * team_games["won"]

    group_keys = [team_games["team_id"], team_games["season"]]

    # Subtract this game's contribution to get the PREGAME record.
    games_before = (
        venue_games.groupby(group_keys).cumsum() - venue_games
    )
    wins_before = (
        venue_wins.groupby(group_keys).cumsum() - venue_wins
    )

    team_games[f"{venue}_wins_before"] = wins_before
    team_games[f"{venue}_losses_before"] = games_before - wins_before

    record_column = (
        "record_at_home" if venue == "home" else "record_as_visitor"
    )
    team_games[record_column] = wins_before / games_before.where(
        games_before > 0
    )

In [39]:
features = [
    "season_win_percentage",
    "win_percentage_last_five_games",
    "rest_days_since_last_game",
    "average_point_differential",
    "points_scored_per_game",
    "points_scored_last_five_games",
    "points_allowed_per_game",
    "points_allowed_last_five_games",
    "record_at_home",
    "record_as_visitor"
]

# Select only identifiers and pregame features.
# Exclude current-game scores/results to avoid leakage.
team_features = team_games[["id", "team_id"] + features].copy()

home_features = team_features.rename(columns={
    "team_id": "home_team_id",
    **{column: f"home_{column}" for column in features},
})

visitor_features = team_features.rename(columns={
    "team_id": "visitor_team_id",
    **{column: f"visitor_{column}" for column in features},
})

nba_model_data = nba_data.merge(
    home_features,
    on=["id", "home_team_id"],
    how="left",
    validate="one_to_one",
).merge(
    visitor_features,
    on=["id", "visitor_team_id"],
    how="left",
    validate="one_to_one",
)

In [40]:
home_team = 0
nba_model_data["home_win"] = (nba_data["home_team_score"] > nba_data["visitor_team_score"]).astype(int)


In [41]:
nba_model_data["last5_win_pct_diff"] = (
    nba_model_data["home_win_percentage_last_five_games"]
    - nba_model_data["visitor_win_percentage_last_five_games"]
)

nba_model_data["season_point_margin_diff"] = (
    nba_model_data["home_average_point_differential"]
    - nba_model_data["visitor_average_point_differential"]
)

nba_model_data["last5_point_margin_diff"] = (
    (nba_model_data["home_points_scored_last_five_games"]
     - nba_model_data["home_points_allowed_last_five_games"])
    - (nba_model_data["visitor_points_scored_last_five_games"]
       - nba_model_data["visitor_points_allowed_last_five_games"])
)

nba_model_data["rest_days_diff"] = (
    nba_model_data["home_rest_days_since_last_game"]
    - nba_model_data["visitor_rest_days_since_last_game"]
)

In [42]:
nba_model_data.head()

,id,date,season,home_team_id,home_team_name,visitor_team_id,visitor_team_name,home_team_score,visitor_team_score,postseason,...,visitor_points_scored_last_five_games,visitor_points_allowed_per_game,visitor_points_allowed_last_five_games,visitor_record_at_home,visitor_record_as_visitor,home_win,last5_win_pct_diff,season_point_margin_diff,last5_point_margin_diff,rest_days_diff
0,21717856,2026-10-20,2026,20,New York Knicks,23,Philadelphia 76ers,100,106,False,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
1,21717857,2026-10-20,2026,27,San Antonio Spurs,21,Oklahoma City Thunder,104,116,False,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
2,21717858,2026-10-21,2026,16,Miami Heat,18,Minnesota Timberwolves,106,131,False,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
3,21717859,2026-10-21,2026,14,Los Angeles Lakers,10,Golden State Warriors,127,105,False,...,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
4,21717922,2026-10-21,2026,22,Orlando Magic,1,Atlanta Hawks,125,132,False,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


In [43]:
nba_model_data.columns

Index(['id', 'date', 'season', 'home_team_id', 'home_team_name',
       'visitor_team_id', 'visitor_team_name', 'home_team_score',
       'visitor_team_score', 'postseason', 'home_season_win_percentage',
       'home_win_percentage_last_five_games', 'home_rest_days_since_last_game',
       'home_average_point_differential', 'home_points_scored_per_game',
       'home_points_scored_last_five_games', 'home_points_allowed_per_game',
       'home_points_allowed_last_five_games', 'home_record_at_home',
       'home_record_as_visitor', 'visitor_season_win_percentage',
       'visitor_win_percentage_last_five_games',
       'visitor_rest_days_since_last_game',
       'visitor_average_point_differential', 'visitor_points_scored_per_game',
       'visitor_points_scored_last_five_games',
       'visitor_points_allowed_per_game',
       'visitor_points_allowed_last_five_games', 'visitor_record_at_home',
       'visitor_record_as_visitor', 'home_win', 'last5_win_pct_diff',
       'season_point_mar

In [73]:
nba_model_features = [
     "home_season_win_percentage",
     "visitor_season_win_percentage",
    "season_point_margin_diff",
    "last5_point_margin_diff",
    "rest_days_diff",
    "last5_win_pct_diff",
    "home_record_at_home",
    "visitor_record_as_visitor",
]  


In [74]:
# drop NaNs

In [75]:
model_data = nba_model_data.dropna(subset=nba_model_features).copy()

In [76]:
#model_data.head()

In [82]:
#Split the data to train and validate

train = model_data[model_data["season"] < 2026]
validation = model_data[model_data["season"] == 2026]

X_train = train[nba_model_features]
y_train = train["home_win"]

X_validation = validation[nba_model_features]
y_validation = validation["home_win"]

In [83]:
# the tree from last week, now with updated features

In [84]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=30,
    random_state=42,
)

model.fit(X_train, y_train)


predictions = model.predict(X_validation)

model_accuracy = accuracy_score(y_validation, predictions)
baseline_accuracy = y_validation.mean()

print(f"Decision tree accuracy: {model_accuracy:.2%}")
print(f"Always-home-win baseline: {baseline_accuracy:.2%}")

print(
    "Training accuracy:",
    f"{accuracy_score(y_train, model.predict(X_train)):.2%}"
)
print(
    "Validation accuracy:",
    f"{accuracy_score(y_validation, model.predict(X_validation)):.2%}"
)
print("Always-home baseline:", f"{y_validation.mean():.2%}")

Decision tree accuracy: 64.13%
Always-home-win baseline: 52.17%
Training accuracy: 65.61%
Validation accuracy: 64.13%
Always-home baseline: 52.17%


In [86]:
from sklearn.metrics import mean_absolute_error

def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeClassifier(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return(mae)

max_nodes = [25, 30, 32]

for node in max_nodes:
    my_mae = get_mae(node, X_train, X_validation, y_train, y_validation)
    print(f"Max leaf nodes: {node}\tMAE: {my_mae:.4f}")

Max leaf nodes: 25	MAE: 0.3370
Max leaf nodes: 30	MAE: 0.3152
Max leaf nodes: 32	MAE: 0.3261


In [87]:
# OK now new stuff: Random forest 

In [89]:
from sklearn.ensemble import RandomForestClassifier

In [91]:
forest_model = RandomForestClassifier(random_state=1)
forest_model.fit(X_train, y_train)
nba_pred = forest_model.predict(X_validation)
print(mean_absolute_error(y_validation, nba_pred))

0.30434782608695654


In [92]:
#MAE of .304, shows improvement from last week

,id,date,season,home_team_id,home_team_name,visitor_team_id,visitor_team_name,home_team_score,visitor_team_score,postseason,...,visitor_points_scored_last_five_games,visitor_points_allowed_per_game,visitor_points_allowed_last_five_games,visitor_record_at_home,visitor_record_as_visitor,home_win,last5_win_pct_diff,season_point_margin_diff,last5_point_margin_diff,rest_days_diff
0,21717856,2026-10-20,2026,20,New York Knicks,23,Philadelphia 76ers,100,106,False,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
1,21717857,2026-10-20,2026,27,San Antonio Spurs,21,Oklahoma City Thunder,104,116,False,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
2,21717858,2026-10-21,2026,16,Miami Heat,18,Minnesota Timberwolves,106,131,False,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
3,21717859,2026-10-21,2026,14,Los Angeles Lakers,10,Golden State Warriors,127,105,False,...,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
4,21717922,2026-10-21,2026,22,Orlando Magic,1,Atlanta Hawks,125,132,False,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,21718090,2026-11-11,2026,25,Portland Trail Blazers,21,Oklahoma City Thunder,118,108,False,...,117.2,105.909091,104.6,1.00,0.5,1,-0.2,1.545455,-10.0,0.0
4128,21717855,2026-10-20,2026,9,Detroit Pistons,2,Boston Celtics,121,113,False,...,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
4129,21717969,2026-10-26,2026,29,Utah Jazz,15,Memphis Grizzlies,105,110,False,...,NaN,107.666667,NaN,1.00,1.0,0,NaN,-20.666667,NaN,0.0
4130,21718020,2026-11-02,2026,18,Minnesota Timberwolves,17,Milwaukee Bucks,112,100,False,...,112.4,126.166667,125.8,0.25,0.0,1,0.2,24.452381,19.8,0.0


,id,date,season,home_team_id,home_team_name,visitor_team_id,visitor_team_name,home_team_score,visitor_team_score,postseason
0,21717856,2026-10-20,2026,20,New York Knicks,23,Philadelphia 76ers,100,106,False
1,21717857,2026-10-20,2026,27,San Antonio Spurs,21,Oklahoma City Thunder,104,116,False
2,21717858,2026-10-21,2026,16,Miami Heat,18,Minnesota Timberwolves,106,131,False
3,21717859,2026-10-21,2026,14,Los Angeles Lakers,10,Golden State Warriors,127,105,False
4,21717922,2026-10-21,2026,22,Orlando Magic,1,Atlanta Hawks,125,132,False
5,21717923,2026-10-21,2026,30,Washington Wizards,17,Milwaukee Bucks,128,116,False
6,21717924,2026-10-21,2026,3,Brooklyn Nets,4,Charlotte Hornets,103,110,False
7,21717926,2026-10-21,2026,28,Toronto Raptors,5,Chicago Bulls,135,92,False
8,21717927,2026-10-21,2026,15,Memphis Grizzlies,29,Utah Jazz,123,103,False
9,21717928,2026-10-21,2026,19,New Orleans Pelicans,12,Indiana Pacers,120,123,False
